In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
ROOT=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'code').is_dir() and (p/'data').is_dir())
OUT=ROOT/'deliverables/Figure4ab_colors_20260908'
OUT.mkdir(parents=True,exist_ok=True)
f34=json.loads((ROOT/'data/reported_stats/figure34_render_stats_v1.json').read_text())['figure4']
drivers=json.loads((ROOT/'data/interim/figure5_structure_score_drivers_v2/figure5_score_drivers_v2.json').read_text())
# Keep evidence-class colours consistent with the manuscript, using muted fills.
PALETTE={'random':'#9AA4AD','direct':'#C39755','cell':'#6F91AF',
         'complex':'#9284AC','structural':'#C87869','line':'#B96556',
         'ink':'#26323C','grid':'#E3E8EC'}
LAYOUT={'width_mm':88,'height_mm':82,'left':.20,'right':.97,'bottom':.27,'top':.81}
mpl.rcParams.update({'font.family':'Arial','font.size':7,'font.weight':'normal',
    'axes.titleweight':'normal','axes.labelweight':'normal','axes.labelsize':7,
    'xtick.labelsize':6,'ytick.labelsize':6,'axes.linewidth':.65,
    'pdf.fonttype':42,'svg.fonttype':'none','savefig.bbox':None})
def canvas(label,title):
    fig=plt.figure(figsize=(LAYOUT['width_mm']/25.4,LAYOUT['height_mm']/25.4))
    ax=fig.add_axes([LAYOUT['left'],LAYOUT['bottom'],LAYOUT['right']-LAYOUT['left'],LAYOUT['top']-LAYOUT['bottom']])
    fig.text(.05,.96,label,fontsize=10,va='top')
    ax.set_title(title,loc='left',fontsize=7.5,pad=9)
    ax.spines['top'].set_visible(False);ax.spines['right'].set_visible(False)
    ax.grid(axis='y',color=PALETTE['grid'],lw=.5);ax.set_axisbelow(True)
    ax.tick_params(width=.65,length=3)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    return fig,ax
def export(fig,stem):
    for ext in ['pdf','svg','png']:fig.savefig(OUT/f'{stem}.{ext}',dpi=600)
    plt.show()
    plt.close(fig)


In [ ]:
# Figure 4a: unchanged fractions, denominators and stored confidence intervals.
fig,ax=canvas('a','Score regimes depend on\nevidence semantics')
order=['Random A–R','Reconstituted direct','Cell binary / proximity','Co-complex association','Structural clean non-contact']
labels=['Random\nunlabeled','Reconstituted\ndirect','Cell binary /\nproximity','Co-complex\nassociation','Assembly\nnoncontact']
colors=[PALETTE[k] for k in ['random','direct','cell','complex','structural']]
rows=[f34['groups'][key] for key in order]
p=np.array([r['frac_ge_0.5'] for r in rows]);ci=np.array([r['frac_ge_0.5_ci95'] for r in rows])
x=np.arange(5)
ax.bar(x,p,color=colors,width=.68,zorder=2)
ax.errorbar(x,p,yerr=[p-ci[:,0],ci[:,1]-p],fmt='none',ecolor=PALETTE['ink'],capsize=2,lw=.8,zorder=3)
for i,r in enumerate(rows):ax.text(i,ci[i,1]+.025,f"{p[i]:.1%}\nn={r['n']:,}",ha='center',va='bottom',fontsize=5.7)
ax.axhline(.5,color='#8B969E',ls=(0,(3,2)),lw=.7)
ax.set_xticks(x);ax.set_xticklabels(labels)
ax.set(ylabel='Fraction with predicted score ≥0.5',ylim=(0,.9))
pd.DataFrame({'group':order,'n':[r['n'] for r in rows],'fraction':p,'lo':ci[:,0],'hi':ci[:,1],'color':colors}).to_csv(OUT/'Figure4a_source.tsv',sep='\t',index=False)
export(fig,'Figure4a')


In [ ]:
# Figure 4b: same structural-cohort colour family as the last bar in 4a.
fig,ax=canvas('b','Training familiarity')
bins=['0-20','20-40','40-60','60-80','80-100']
block=drivers['5a_training_exposure']['by_endpoint_train_familiarity']
p=np.array([block[k]['frac_ge_0.5'] for k in bins]);ci=np.array([block[k]['ci95'] for k in bins])
x=np.arange(5)
ax.plot(x,p,color=PALETTE['line'],marker='o',ms=4,lw=1.5,
        markerfacecolor=PALETTE['structural'],markeredgecolor=PALETTE['line'])
ax.errorbar(x,p,yerr=[p-ci[:,0],ci[:,1]-p],fmt='none',ecolor=PALETTE['line'],capsize=2,lw=.8)
ax.axhline(.5,color='#8B969E',ls=(0,(3,2)),lw=.7)
ax.set_xticks(x);ax.set_xticklabels(['<20 /\nno hit','20–40','40–60','60–80','80–100'])
ax.set(ylabel='Fraction score ≥0.5',ylim=(.48,1.02),xlabel='Minimum endpoint identity to\npositive-training proteins (%)')
pd.DataFrame({'bin':bins,'fraction':p,'lo':ci[:,0],'hi':ci[:,1]}).to_csv(OUT/'Figure4b_source.tsv',sep='\t',index=False)
export(fig,'Figure4b')
